# Speech Emotion Recognition — Phase 2: Features & Data
**Week 2 of the roadmap** · Attention-Based CNN-LSTM with Optimal Feature Selection

Phase 1 (signals, FFT, STFT/spectrogram) is done. This week turns raw audio into **numbers a model can learn from**, and builds the **dataset** for everything that follows.

| Day | Roadmap stage | What you build |
|---|---|---|
| 1 | Speech Feature Extraction | MFCC **from scratch** (NumPy) and verify it against Librosa |
| 2 | Speech Feature Extraction | Mel scale, filter bank, Mel Spectrogram |
| 3 | Speech Feature Extraction | Delta / Delta-Delta MFCC and Chroma |
| 4 | Dataset Creation | Parse RAVDESS + TESS into one labelled table |
| 5 | Dataset Creation | Extraction pipeline: `Samples × Features` and `Samples × MFCC × Time` |
| 6 | Dataset Creation | Quality checks, speaker-independent split, normalisation |
| 7 | Review | Feature-combination tensors for Phase 3, save to disk, sanity check |

**Datasets.** Put RAVDESS in `data/RAVDESS/` (e.g. `data/RAVDESS/Actor_01/*.wav`) and TESS in `data/TESS/`.
If neither folder exists, the notebook **automatically generates a small synthetic demo dataset** with RAVDESS-style file names so every cell still runs. Swap in the real data and re-run top to bottom.

* RAVDESS: https://zenodo.org/records/1188976 (Audio_Speech_Actors_01-24.zip)
* TESS: https://tspace.library.utoronto.ca/handle/1807/24487 (also on Kaggle)

**Install:** `pip install librosa numpy scipy pandas matplotlib scikit-learn soundfile joblib`

---
# Day 1 — Setup and MFCC from scratch
**Goal:** understand every step that turns a waveform into MFCCs, by implementing them yourself.

In [ ]:
import os, re, glob, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display
import scipy.fft, scipy.signal
import soundfile as sf
from IPython.display import Audio, display
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100

# ---- global feature settings (keep these fixed for the whole project) ----
SR       = 22050      # sampling rate (Hz)
N_FFT    = 2048       # FFT window length (samples)  ~ 93 ms
HOP      = 512        # hop between frames (samples) ~ 23 ms
N_MFCC   = 40         # MFCC coefficients
N_MELS   = 64         # mel filters / mel-spectrogram bins
DURATION = 3.0        # every clip is padded/cropped to 3 s
MAX_FRAMES = 1 + int(DURATION * SR) // HOP   # time frames per clip (= 130)

RAVDESS_DIR, TESS_DIR, DEMO_DIR, OUT_DIR = "data/RAVDESS", "data/TESS", "data/demo", "features"
os.makedirs(OUT_DIR, exist_ok=True)

EMOTIONS = ["happy", "sad", "angry", "neutral", "fear", "disgust", "surprise", "calm"]
LABEL2ID = {e: i for i, e in enumerate(EMOTIONS)}
print("frames per clip:", MAX_FRAMES)

### Dataset helpers (file-name parsing + demo-data generator)
* **RAVDESS** file name = `modality-vocal-emotion-intensity-statement-repetition-actor.wav`, e.g. `03-01-06-01-02-01-12.wav` → emotion `06` = fear, actor 12.
* **TESS** has two actresses (OAF, YAF); the emotion is the last word of the file name (`OAF_back_angry.wav`). `ps` = pleasant surprise.

In [ ]:
RAV_MAP = {"01":"neutral","02":"calm","03":"happy","04":"sad",
           "05":"angry","06":"fear","07":"disgust","08":"surprise"}
RAV_RE  = re.compile(r"^(\d\d)-(\d\d)-(\d\d)-(\d\d)-(\d\d)-(\d\d)-(\d\d)\.wav$")

def parse_any(path):
    # Return dataset / emotion / speaker for a RAVDESS or TESS file
    name = os.path.basename(path)
    m = RAV_RE.match(name)
    if m:
        mod, vc, emo, inten, stmt, rep, actor = m.groups()
        return dict(dataset="RAVDESS", emotion=RAV_MAP[emo], speaker=f"R{actor}", intensity=int(inten))
    parts = os.path.splitext(name)[0].split("_")
    emo = parts[-1].lower()
    if emo.startswith("ps") or "surpris" in emo or "pleasant" in emo: emo = "surprise"
    if emo == "fearful": emo = "fear"
    return dict(dataset="TESS", emotion=emo, speaker=f"T{parts[0]}", intensity=np.nan)

def find_wavs(root):
    return sorted(glob.glob(os.path.join(root, "**", "*.wav"), recursive=True))

# ---- synthetic demo speech so the notebook runs without the real data ----
_F0 = dict(happy=260, sad=170, angry=225, neutral=200, fear=300, disgust=180, surprise=340, calm=190)
_VIB = dict(happy=5, sad=2, angry=7, neutral=3, fear=9, disgust=3, surprise=6, calm=2)
_PULSE = dict(happy=4.5, sad=2.0, angry=5.5, neutral=3.0, fear=6.0, disgust=2.5, surprise=4.0, calm=2.2)
_NOISE = dict(happy=.01, sad=.01, angry=.05, neutral=.01, fear=.03, disgust=.04, surprise=.01, calm=.005)

def synth_clip(emotion, actor, seed, sr=SR):
    rng = np.random.default_rng(seed)
    dur = rng.uniform(1.8, 2.8)
    t = np.arange(int(dur * sr)) / sr
    f0 = _F0[emotion] * (1.0 + 0.12 * (actor % 4 - 1.5) / 1.5) * rng.uniform(0.97, 1.03)
    phase = 2*np.pi*np.cumsum(f0 * (1 + 0.03*np.sin(2*np.pi*_VIB[emotion]*t)) / sr)
    y = sum(np.sin(k*phase) / k**1.2 for k in range(1, 9))
    env = 0.5 * (1 + np.sin(2*np.pi*_PULSE[emotion]*t - np.pi/2)) ** 1.5
    y = y * env + _NOISE[emotion] * rng.standard_normal(len(t))
    y = np.concatenate([np.zeros(int(sr*rng.uniform(.1, .4))), y, np.zeros(int(sr*.1))])
    return (0.8 * y / np.max(np.abs(y))).astype(np.float32)

def make_demo_dataset(root=DEMO_DIR, n_actors=6):
    code_of = {v: k for k, v in RAV_MAP.items()}
    for a in range(1, n_actors + 1):
        d = os.path.join(root, f"Actor_{a:02d}"); os.makedirs(d, exist_ok=True)
        for emo in EMOTIONS:
            for stmt in (1, 2):
                for rep in (1, 2):
                    fn = f"03-01-{code_of[emo]}-01-{stmt:02d}-{rep:02d}-{a:02d}.wav"
                    sf.write(os.path.join(d, fn), synth_clip(emo, a, a*1000 + int(code_of[emo])*10 + stmt*2 + rep), SR)

ravdess_files, tess_files = find_wavs(RAVDESS_DIR), find_wavs(TESS_DIR)
USING_DEMO = False
if not ravdess_files and not tess_files:
    USING_DEMO = True
    if not find_wavs(DEMO_DIR): make_demo_dataset()
    ravdess_files = find_wavs(DEMO_DIR)
ALL_FILES = ravdess_files + tess_files
print(f"RAVDESS: {len(ravdess_files)} files | TESS: {len(tess_files)} files | demo mode: {USING_DEMO}")

In [ ]:
def first_by_emotion(files):
    out = {}
    for f in files:
        e = parse_any(f)["emotion"]
        if e in LABEL2ID and e not in out: out[e] = f
    return out

SAMPLES = first_by_emotion(ALL_FILES)
y, _ = librosa.load(SAMPLES["happy"], sr=SR)
print(f"samples: {len(y)}  duration: {len(y)/SR:.2f}s")
display(Audio(y, rate=SR))
plt.figure(figsize=(10, 2.5)); librosa.display.waveshow(y, sr=SR); plt.title("Waveform — 'happy' sample"); plt.tight_layout(); plt.show()

### The MFCC pipeline (5 steps)
```
waveform → [1 frame + window] → [2 FFT → power] → [3 mel filter bank] → [4 log] → [5 DCT] → MFCC
```
1. **Frame + window** – speech is non-stationary, so cut it into short overlapping frames and multiply each by a Hann window (reduces edge artefacts).
2. **FFT → power spectrum** – frequency content of each frame (`|X|²`).
3. **Mel filter bank** – triangular filters spaced on the *mel* scale (dense at low frequencies, sparse at high), mimicking human hearing.
4. **Log** – humans perceive loudness logarithmically; also compresses dynamic range.
5. **DCT** – decorrelates the neighbouring mel bands. Keeping only the first ~40 coefficients keeps the **spectral envelope** (vocal-tract shape) and drops fine detail (pitch harmonics).

Below is each step implemented with plain NumPy.

In [ ]:
def hz_to_mel(f):  return 2595.0 * np.log10(1.0 + f / 700.0)       # HTK mel formula
def mel_to_hz(m):  return 700.0 * (10.0 ** (m / 2595.0) - 1.0)

def mel_filterbank(sr, n_fft, n_mels, fmin=0.0, fmax=None):
    # Triangular filters equally spaced on the mel scale -> matrix (n_mels, 1 + n_fft/2)
    fmax = fmax or sr / 2
    fft_freqs = np.linspace(0, sr / 2, 1 + n_fft // 2)
    hz_pts = mel_to_hz(np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_mels + 2))
    fdiff = np.diff(hz_pts)
    ramps = hz_pts[:, None] - fft_freqs[None, :]
    W = np.zeros((n_mels, len(fft_freqs)))
    for i in range(n_mels):
        lower = -ramps[i] / fdiff[i]          # rising edge
        upper = ramps[i + 2] / fdiff[i + 1]   # falling edge
        W[i] = np.maximum(0, np.minimum(lower, upper))
    W *= (2.0 / (hz_pts[2:n_mels + 2] - hz_pts[:n_mels]))[:, None]   # area-normalise (Slaney) so every filter has equal energy
    return W

def mfcc_from_scratch(y, sr=SR, n_fft=N_FFT, hop=HOP, n_mels=N_MELS, n_mfcc=N_MFCC):
    # 1) framing + Hann window (reflect-pad so frames are centred like librosa)
    ypad = np.pad(y, n_fft // 2, mode="reflect")
    n_frames = 1 + (len(ypad) - n_fft) // hop
    idx = np.arange(n_fft)[None, :] + hop * np.arange(n_frames)[:, None]
    frames = ypad[idx] * scipy.signal.get_window("hann", n_fft, fftbins=True)
    # 2) FFT -> power spectrum  (freq_bins, time)
    power = (np.abs(np.fft.rfft(frames, axis=1)) ** 2).T
    # 3) mel filter bank
    mel = mel_filterbank(sr, n_fft, n_mels) @ power
    # 4) log (decibel) compression
    log_mel = librosa.power_to_db(mel)
    # 5) DCT-II, keep first n_mfcc coefficients
    mfcc = scipy.fft.dct(log_mel, axis=0, type=2, norm="ortho")[:n_mfcc]
    return dict(power=power, mel=mel, log_mel=log_mel, mfcc=mfcc)

steps = mfcc_from_scratch(y)
ref = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP,
                           n_mels=N_MELS, htk=True)
print("my MFCC shape     :", steps["mfcc"].shape, "(coefficients, frames)")
print("librosa MFCC shape:", ref.shape)
print("max |difference|  :", np.abs(steps["mfcc"] - ref).max())
assert np.allclose(steps["mfcc"], ref, atol=1e-3), "scratch MFCC should match librosa"

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
titles = ["2) Power spectrum (dB)", "3) Mel spectrum (dB)", "4) Log-mel", "5) MFCC (after DCT)"]
data = [librosa.power_to_db(steps["power"]), librosa.power_to_db(steps["mel"]), steps["log_mel"], steps["mfcc"]]
for a, d, t in zip(ax, data, titles):
    im = a.imshow(d, origin="lower", aspect="auto", cmap="magma"); a.set_title(t); a.set_xlabel("frame")
    fig.colorbar(im, ax=a)
ax[0].set_ylabel("bin / coefficient"); plt.tight_layout(); plt.show()

**Day 1 takeaways**
* Your from-scratch MFCC matches Librosa to numerical precision — you now know exactly what `librosa.feature.mfcc` does.
* MFCC output shape is `(n_coefficients, n_frames)`: rows = *what* (spectral shape), columns = *when* (time).
* Coefficient 0 is mostly overall energy; low coefficients describe the broad spectral envelope; high ones are fine detail/noise.

---
# Day 2 — Mel scale, filter bank and Mel Spectrogram
**Goal:** see *why* the mel scale is used and learn the feature that CNNs consume most naturally.

In [ ]:
f = np.linspace(0, SR / 2, 1000)
W = mel_filterbank(SR, N_FFT, N_MELS)
fft_f = np.linspace(0, SR / 2, 1 + N_FFT // 2)

fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
ax[0].plot(f, hz_to_mel(f)); ax[0].set_xlabel("Frequency (Hz)"); ax[0].set_ylabel("Mel")
ax[0].set_title("Hz → Mel: compresses high frequencies")
for i in range(0, N_MELS, 4): ax[1].plot(fft_f, W[i])
ax[1].set_xlim(0, 8000); ax[1].set_xlabel("Frequency (Hz)")
ax[1].set_title(f"Every 4th of {N_MELS} mel filters — narrow at low Hz, wide at high Hz")
plt.tight_layout(); plt.show()

In [ ]:
S_lin = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)) ** 2
S_mel = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
S_mel_db = librosa.power_to_db(S_mel, ref=np.max)

fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
librosa.display.specshow(librosa.power_to_db(S_lin, ref=np.max), sr=SR, hop_length=HOP, x_axis="time", y_axis="hz", ax=ax[0]); ax[0].set_title(f"Linear spectrogram {S_lin.shape}")
librosa.display.specshow(S_mel_db, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=ax[1]); ax[1].set_title(f"Mel spectrogram (dB) {S_mel_db.shape}")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 6.5))
for ax, emo in zip(axes.ravel(), EMOTIONS):
    if emo not in SAMPLES: ax.axis("off"); continue
    yy, _ = librosa.load(SAMPLES[emo], sr=SR)
    M = librosa.power_to_db(librosa.feature.melspectrogram(y=yy, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS), ref=np.max)
    librosa.display.specshow(M, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=ax); ax.set_title(emo)
plt.suptitle("Mel spectrogram per emotion"); plt.tight_layout(); plt.show()

**Day 2 takeaways**
* **Mel spectrogram** = steps 1-4 of the MFCC pipeline (no DCT). It keeps all `n_mels` bands → richer, image-like input for a **CNN**.
* **MFCC** = Mel spectrogram + DCT → compact and decorrelated, ideal for classical ML and sequence models.
* Always convert power → dB (`power_to_db`); raw power has a huge dynamic range that hurts training.

---
# Day 3 — Delta MFCC and Chroma
**Goal:** add *dynamics* (how speech changes) and *pitch-class / harmonic* information.

**Delta MFCC** — the rate of change of each coefficient over time. Emotion lives in dynamics (angry speech changes fast, sad speech slowly).
For a window of ±N frames: `d_t = Σ n·(c[t+n] − c[t−n]) / (2·Σ n²)`.  **Delta-delta** = delta of delta (acceleration).

In [ ]:
def delta_from_scratch(C, N=2):
    # Regression-based delta along the time axis (axis=1), edges replicated
    Cp = np.pad(C, ((0, 0), (N, N)), mode="edge")
    den = 2 * sum(n * n for n in range(1, N + 1))
    return sum(n * (Cp[:, N + n: Cp.shape[1] - N + n] - Cp[:, N - n: Cp.shape[1] - N - n]) for n in range(1, N + 1)) / den

mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
d1_mine = delta_from_scratch(mfcc, N=2)
d1_lib  = librosa.feature.delta(mfcc, width=5, mode="nearest")
print("max |diff| vs librosa delta:", np.abs(d1_mine - d1_lib).max())
assert np.allclose(d1_mine, d1_lib, atol=1e-4)

d1 = librosa.feature.delta(mfcc); d2 = librosa.feature.delta(mfcc, order=2)
fig, ax = plt.subplots(1, 3, figsize=(17, 3.6))
for a, d, t in zip(ax, [mfcc, d1, d2], ["MFCC", "Delta MFCC", "Delta-Delta MFCC"]):
    im = a.imshow(d, origin="lower", aspect="auto", cmap="coolwarm"); a.set_title(t); a.set_xlabel("frame"); fig.colorbar(im, ax=a)
ax[0].set_ylabel("coefficient"); plt.tight_layout(); plt.show()

**Chroma** — folds all frequencies into the **12 pitch classes** (C, C#, … B). Captures pitch/harmonic content (intonation patterns), shape `(12, frames)`.

In [ ]:
chroma = librosa.feature.chroma_stft(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP)
print("chroma shape:", chroma.shape)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
librosa.display.specshow(chroma, sr=SR, hop_length=HOP, x_axis="time", y_axis="chroma", ax=ax[0]); ax[0].set_title("Chroma (happy)")
yy, _ = librosa.load(SAMPLES["sad"], sr=SR)
librosa.display.specshow(librosa.feature.chroma_stft(y=yy, sr=SR, n_fft=N_FFT, hop_length=HOP), sr=SR, hop_length=HOP, x_axis="time", y_axis="chroma", ax=ax[1]); ax[1].set_title("Chroma (sad)")
plt.tight_layout(); plt.show()

print("\nFeature cheat-sheet (all share the same time axis):")
print(pd.DataFrame({"feature": ["MFCC", "Delta MFCC", "Delta-Delta", "Mel spectrogram", "Chroma"],
                    "rows": [N_MFCC, N_MFCC, N_MFCC, N_MELS, 12],
                    "captures": ["spectral envelope", "speed of change", "acceleration", "full band energy (CNN input)", "pitch classes / harmonics"]}).to_string(index=False))

---
# Day 4 — Dataset creation (part 1): the labelled table
**Goal:** one clean `DataFrame` describing every audio file: path, dataset, emotion, numeric label, speaker.

* **Primary:** RAVDESS (24 actors, 8 emotions incl. *calm*).
* **Secondary:** TESS (2 speakers, 7 emotions — has *pleasant surprise* but **no calm**).

In [ ]:
rows = []
for p in ALL_FILES:
    info = parse_any(p)
    if info["emotion"] not in LABEL2ID: continue          # skip anything unexpected
    rows.append(dict(path=p, **info, label=LABEL2ID[info["emotion"]]))
df = pd.DataFrame(rows)

# duration of every clip (header only, fast)
df["duration"] = [sf.info(p).duration for p in df["path"]]
df["speaker_id"] = df["speaker"]
print(df.shape); display(df.head())

In [ ]:
print("Per dataset:"); display(df.groupby("dataset").size().to_frame("clips"))
counts = pd.crosstab(df["emotion"], df["dataset"]).reindex(EMOTIONS).fillna(0).astype(int)
display(counts)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
counts.plot.bar(stacked=True, ax=ax[0]); ax[0].set_title("Class distribution"); ax[0].set_ylabel("clips")
ax[1].hist(df["duration"], bins=30); ax[1].axvline(DURATION, color="r", ls="--", label=f"target {DURATION}s")
ax[1].set_title("Clip duration (s)"); ax[1].legend(); plt.tight_layout(); plt.show()
print("speakers:", df["speaker_id"].nunique())

**Check:** are the classes balanced? Imbalance (e.g. RAVDESS *neutral* has half the clips; *calm* only exists in RAVDESS) affects accuracy — in Phase 3 report **F1 / confusion matrix**, not just accuracy.

---
# Day 5 — Dataset creation (part 2): feature extraction pipeline
**Per file:** `load → trim silence → fix length → extract → store → attach label`

Two output formats from the roadmap:
* **Classical ML:** `Samples × Features` (each clip → one vector: mean & std over time of each coefficient)
* **Deep learning:** `Samples × Feature × Time` (keeps the time axis for CNN-LSTM)

In [ ]:
def load_fixed(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)               # remove leading/trailing silence
    target = int(DURATION * SR)
    y = np.pad(y, (0, target - len(y))) if len(y) < target else y[:target]   # fixed length
    peak = np.max(np.abs(y))
    return (y / peak if peak > 0 else y).astype(np.float32)  # peak-normalise loudness

def extract_features(path):
    y = load_fixed(path)
    mfcc   = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    mel    = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS), ref=np.max)
    chroma = librosa.feature.chroma_stft(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP)
    out = dict(mfcc=mfcc, delta=delta, delta2=delta2, mel=mel, chroma=chroma)
    for k, v in out.items(): assert v.shape[1] == MAX_FRAMES, (k, v.shape)
    return {k: v.astype(np.float32) for k, v in out.items()}

t = extract_features(df.path.iloc[0])
print({k: v.shape for k, v in t.items()})

In [ ]:
from joblib import Parallel, delayed
t0 = time.time()
results = Parallel(n_jobs=-1)(delayed(extract_features)(p) for p in df["path"])
print(f"extracted {len(results)} clips in {time.time()-t0:.1f}s")

X = {k: np.stack([r[k] for r in results]) for k in ["mfcc", "delta", "delta2", "mel", "chroma"]}
y_all = df["label"].values.astype(np.int64)
for k, v in X.items(): print(f"X[{k!r:9}] shape = {v.shape}   (Samples × Feature × Time)")

In [ ]:
def flat_stats(M):
    # (N, F, T) -> (N, 2F): mean and std over time of each coefficient
    return np.concatenate([M.mean(axis=2), M.std(axis=2)], axis=1)

X_flat_mean40 = X["mfcc"].mean(axis=2)                       # roadmap example: Samples × 40
X_flat = flat_stats(X["mfcc"])                               # Samples × 80
print("classical ML  (mean only):", X_flat_mean40.shape)
print("classical ML  (mean+std) :", X_flat.shape)
print("deep learning (sequence) :", X["mfcc"].shape)

---
# Day 6 — Quality checks, speaker-independent split, normalisation

In [ ]:
# ---- 1. sanity checks ----
for k, v in X.items():
    print(f"{k:7} NaN: {np.isnan(v).sum()}  Inf: {np.isinf(v).sum()}  min {v.min():9.2f}  max {v.max():8.2f}  mean {v.mean():8.2f}")
assert not any(np.isnan(v).any() for v in X.values())

# ---- 2. does the feature actually differ between emotions? (mean MFCC per emotion) ----
M = np.stack([X["mfcc"][y_all == i].mean(axis=(0, 2)) for i in range(len(EMOTIONS)) if (y_all == i).any()])
present = [e for i, e in enumerate(EMOTIONS) if (y_all == i).any()]
plt.figure(figsize=(12, 3.5)); plt.imshow(M, aspect="auto", cmap="coolwarm"); plt.colorbar()
plt.yticks(range(len(present)), present); plt.xlabel("MFCC coefficient"); plt.title("Mean MFCC per emotion"); plt.tight_layout(); plt.show()

### Speaker-independent split
If the same speaker appears in train **and** test, the model can memorise the *voice* rather than the *emotion*, inflating accuracy. So we split **by speaker** (`GroupShuffleSplit`): ~70 % train / 15 % val / 15 % test speakers.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
groups = df["speaker_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainval_idx, test_idx = next(gss.split(np.zeros(len(df)), y_all, groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.18, random_state=42)
tr, va = next(gss2.split(np.zeros(len(trainval_idx)), y_all[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr], trainval_idx[va]

for name, ix in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(f"{name:5}: {len(ix):5d} clips | speakers: {sorted(set(groups[ix]))[:6]}{'...' if len(set(groups[ix]))>6 else ''}")
assert not (set(groups[train_idx]) & set(groups[test_idx])), "speaker leakage!"
assert not (set(groups[train_idx]) & set(groups[val_idx])), "speaker leakage!"

### Normalisation
Scale each coefficient to zero mean / unit variance **using training statistics only** (using val/test statistics would leak information).

In [ ]:
def fit_norm(A, idx):
    return A[idx].mean(axis=(0, 2), keepdims=True), A[idx].std(axis=(0, 2), keepdims=True) + 1e-8

def apply_norm(A, stats):
    return ((A - stats[0]) / stats[1]).astype(np.float32)

stats = {k: fit_norm(X[k], train_idx) for k in X}
X_norm = {k: apply_norm(X[k], stats[k]) for k in X}
print("train mfcc after norm -> mean %.3f, std %.3f" % (X_norm["mfcc"][train_idx].mean(), X_norm["mfcc"][train_idx].std()))
print("test  mfcc after norm -> mean %.3f, std %.3f  (not exactly 0/1 — that's expected)" % (X_norm["mfcc"][test_idx].mean(), X_norm["mfcc"][test_idx].std()))

---
# Day 7 — Feature combinations, saving, sanity check
Phase 3 (*Feature Selection*) compares these combinations. We prepare them now as `(Samples, Features, Time)` tensors by stacking along the feature axis.

| Name | Composition | Features |
|---|---|---|
| `mfcc` | MFCC | 40 |
| `mfcc+delta` | MFCC + Δ | 80 |
| `mfcc+mel` | MFCC + Mel | 104 |
| `mfcc+chroma` | MFCC + Chroma | 52 |
| `all` | MFCC + Δ + ΔΔ + Mel + Chroma | 196 |

In [ ]:
COMBOS = {
    "mfcc":        ["mfcc"],
    "mfcc+delta":  ["mfcc", "delta"],
    "mfcc+mel":    ["mfcc", "mel"],
    "mfcc+chroma": ["mfcc", "chroma"],
    "all":         ["mfcc", "delta", "delta2", "mel", "chroma"],
}
def get_features(name, normalised=True):
    src = X_norm if normalised else X
    return np.concatenate([src[k] for k in COMBOS[name]], axis=1)

print(pd.DataFrame([(n, get_features(n).shape) for n in COMBOS], columns=["combo", "shape (N, F, T)"]).to_string(index=False))

In [ ]:
# ---- save everything Phase 3 needs ----
np.savez_compressed(os.path.join(OUT_DIR, "ser_features.npz"),
    **{f"X_{k}": v for k, v in X.items()},
    X_flat_mean40=X_flat_mean40, X_flat=X_flat, y=y_all,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
    **{f"mean_{k}": stats[k][0] for k in stats}, **{f"std_{k}": stats[k][1] for k in stats})
df.drop(columns=["speaker"]).to_csv(os.path.join(OUT_DIR, "metadata.csv"), index=False)
pd.Series(EMOTIONS).to_csv(os.path.join(OUT_DIR, "labels.csv"), index_label="id", header=["emotion"])
print("saved ->", os.listdir(OUT_DIR))
print("size: %.1f MB" % (os.path.getsize(os.path.join(OUT_DIR, "ser_features.npz")) / 1e6))

In [ ]:
# ---- smoke test: do features and labels line up? (NOT the Phase 3 baseline) ----
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(X_flat[train_idx])
clf = LogisticRegression(max_iter=3000).fit(sc.transform(X_flat[train_idx]), y_all[train_idx])
acc = clf.score(sc.transform(X_flat[test_idx]), y_all[test_idx])
print(f"smoke-test accuracy on unseen speakers: {acc:.2%}  (chance = {1/len(set(y_all)):.0%})")
if USING_DEMO: print("NOTE: demo/synthetic data — numbers are only meaningful on real RAVDESS/TESS.")

## Week 2 checklist
- [x] MFCC implemented from scratch and verified against Librosa
- [x] Mel spectrogram, Delta, Delta-Delta, Chroma extracted
- [x] RAVDESS + TESS parsed into one labelled table
- [x] `Samples × Features` and `Samples × MFCC × Time` datasets built
- [x] Speaker-independent train/val/test split, train-only normalisation
- [x] Everything saved to `features/ser_features.npz`

**Next (Phase 3):** machine-learning baselines (Logistic Regression, Decision Tree, Random Forest, SVM) on `X_flat`, then compare the `COMBOS` above to find the optimal feature set.

**Load later with:**
```python
d = np.load("features/ser_features.npz")
X_mfcc, y = d["X_mfcc"], d["y"]; train_idx, test_idx = d["train_idx"], d["test_idx"]
```